# RAG Evaluation with RAGAS

**Intro to RAG evaluation · The RAGAS metrics · Preparing a dataset · Evaluating a RAG app**

---

### The question this notebook answers

You built a RAG chatbot. Your boss asks:

> *"Is it good?"*

Right now your honest answer is *"I tried about ten questions and they looked fine."*

That is not an answer. It does not scale, it cannot be repeated, and it cannot tell you whether the change you shipped yesterday made things better or worse.

**RAGAS turns "it looked fine" into a number.**

---

# Part 1 - Introduction to RAG Evaluation and RAGAS

### Why evaluating RAG is harder than normal testing

A normal function has one right answer, so testing is easy:

```
   add(2, 2)  ->  4        correct or not, no argument
```

A RAG app has **many** acceptable answers:

```
   "How long do refunds take?"
      ->  "Refunds take 7 business days."
      ->  "You will get your money back within a week of business days."
      ->  "Seven working days."
```

All three are right. `assert answer == expected` is useless here.

---

### The key insight: a RAG app has TWO halves

This is the most important idea in the notebook.

```
                    HALF 1: RETRIEVER                  HALF 2: GENERATOR
                 (find the right documents)        (write an answer from them)

   question ───────────► search ───────────► documents ───────────► LLM ───────────► answer
                            |                                        |
                            |                                        |
                    can fail by bringing              can fail by ignoring good
                    the wrong documents               documents and making things up
```

**They fail in completely different ways, and the fix is completely different.**

| **What you see**                                    | **Which half broke** | **What you fix**                |
| --------------------------------------------------- | -------------------- | ------------------------------- |
| Answer is wrong, and the documents were wrong too   | Retriever            | chunking, embeddings, search, k |
| Answer is wrong, but the right documents were there | Generator            | the prompt, the model           |

A single "accuracy" score cannot tell these apart. That is exactly the gap RAGAS fills: **it scores each half separately.**

---

### So what is RAGAS?

> **RAGAS is a library that scores your RAG answers from 0 to 1, using an LLM as the judge.**

You give it what your app did (question, documents it found, answer it wrote). RAGAS asks a second LLM careful questions about that, and returns numbers.

```
   your RAG output  ──►  RAGAS  ──►  faithfulness      0.95
                          |          answer relevancy  0.88
                     (judge LLM)     context precision 0.50   <- the weak spot
                                     context recall    0.45
```

### Two things to be clear about up front

1. **The judge is an LLM, so evaluation costs money and takes time.** Every metric on every row is at least one LLM call. 5 questions x 4 metrics is 20-plus calls. Start small.
2. **Scores are not exact truth.** They are a consistent, repeatable opinion. Their real value is **comparing** version A against version B, not the absolute number.

---

# Part 2 - Understanding RAGAS Metrics

### The four you should learn first

| **Metric**            | **The question it asks**                              | **Which half it tests** |
| --------------------- | ----------------------------------------------------- | ----------------------- |
| **Faithfulness**      | Is the answer actually supported by the documents?    | Generator               |
| **Answer relevancy**  | Does the answer address the question that was asked?  | Generator               |
| **Context precision** | Were the retrieved documents useful, or mostly noise? | Retriever               |
| **Context recall**    | Did we retrieve everything needed to answer?          | Retriever               |

All four return **0 to 1, higher is better.**

---

### Reading a low score

This table is the reason you run RAGAS at all.

| **Low score**     | **What it means in plain English**                          | **Where to look**                              |
| ----------------- | ----------------------------------------------------------- | ---------------------------------------------- |
| Faithfulness      | The model is **making things up**, even with good documents | the prompt, the model                          |
| Answer relevancy  | The answer wanders or dodges the question                   | the prompt                                     |
| Context precision | You retrieved a lot of junk alongside the useful bit        | chunk size, k, reranking                       |
| Context recall    | The needed information was **never retrieved**              | chunking, embeddings, the documents themselves |

> **Worked example.** Faithfulness 0.4 with context recall 0.95 means the documents were there and the model ignored them. Do not touch your retriever. Fix the prompt.

---

### The split that decides your whole workflow: does it need a reference answer?

A **reference** is a human-written correct answer, also called ground truth. Writing them is slow, so this matters:

| **Metric**                         | **Needs a reference?** |
| ---------------------------------- | ---------------------- |
| Faithfulness                       | **No**                 |
| Answer relevancy                   | **No**                 |
| Context precision (with reference) | Yes                    |
| Context recall                     | Yes                    |
| Factual correctness                | Yes                    |
| Semantic similarity                | Yes                    |

**Why you care:** the two "No" metrics can run on **live production traffic**, where nobody has written a correct answer. The "Yes" metrics need a prepared test set.

So the usual setup is:

```
   test set with references   ->  all metrics       ->  run before each release
   live production traffic    ->  the 2 no-reference metrics  ->  run continuously
```

In [1]:
#!pip install -U ragas langchain-google-genai langchain-community langchain-core python-dotenv

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

# Add this line to your .env file: GOOGLE_API_KEY=your_gemini_api_key
if os.environ.get("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY loaded")
else:
    print("GOOGLE_API_KEY MISSING, add it to your .env file")

GOOGLE_API_KEY loaded


```
GOOGLE_API_KEY loaded
```

### Step 2 - The four field names

A RAGAS row has four fields. **The names are strict.**

| **Field**            | **What goes in it**                                    | **Comes from**  |
| -------------------- | ------------------------------------------------------ | --------------- |
| `user_input`         | the question                                           | you             |
| `retrieved_contexts` | the documents your app found, as a **list of strings** | your retriever  |
| `response`           | the answer your app produced                           | your LLM        |
| `reference`          | the correct answer, written by a human                 | you, in advance |

> **This is the number one place RAGAS code breaks.** Older tutorials use the old names. If you copy code from a blog post and get a confusing error, check this table first.
>
> \| Old name (do not use) | Current name | |---|---| | `question` | `user_input` | | `contexts` | `retrieved_contexts` | | `answer` | `response` | | `ground_truth` | `reference` |

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate

GEMINI_MODEL = "gemini-3.8-flash"

documents = [
    "Refunds are processed within 7 business days for orders placed under 30 days ago.",
    "Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.",
    "Our support team is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel a subscription anytime from Settings. Billing stops at the end of the cycle.",
    "Damaged items must be reported with a photo within 48 hours of delivery.",
]

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant. Answer ONLY using this context: {context}"),
    ("human", "{question}"),
])

print("RAG app ready")

c:\Users\monis\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\auth\transport\grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


RAG app ready


In [2]:
test_questions = [
    "how long do refund take?",
    "How fast is express shipping?",
    "when is the support team available?",
    "How do i cance my subscription"
]

reference_answer = [
    "Refunds are processed within 7 business days.",
    "Express shipping takes 1 business day.",
    "Support is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel anytime from Settings, and billing stops at the end of the cycle.",
]

In [3]:
rows = []

for question, reference in zip(test_questions,reference_answer):

    docs = retriever.invoke(question)

    contexts = []
    for doc in docs:
        contexts.append(doc.page_content)

    context_text = " ".join(contexts)
    messages = prompt.format_messages(context = context_text, question = question)

    answer = llm.invoke(messages).content

    rows.append({
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": reference
    })

print("done")

done


In [4]:
from ragas import EvaluationDataset

dataset = EvaluationDataset.from_list(rows)

print("Dataset is ready: " , len(dataset))

c:\Users\monis\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset is ready:  4


In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import  LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
)

print("judge ready")

judge ready


C:\Users\monis\AppData\Local\Temp\ipykernel_32156\2005917828.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0))
C:\Users\monis\AppData\Local\Temp\ipykernel_32156\2005917828.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(


In [7]:
from ragas.metrics import (
    Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall)

metrics = [
    Faithfulness(), 
    ResponseRelevancy(), 
    LLMContextPrecisionWithReference(), 
    LLMContextRecall()
]

print("done")

done


In [8]:
from ragas import  evaluate

result = evaluate(
    dataset = dataset,
    metrics = metrics,
    llm = judge_llm,
    embeddings = judge_embeddings
)

print(result)

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.8-flash
Please retry in 31.495756523s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.8-flash"
  }
  quota_dimensions {
    key: "location"
 

{'faithfulness': nan, 'answer_relevancy': 0.9828, 'llm_context_precision_with_reference': nan, 'context_recall': 1.0000}


```
```
```